# 3.27 — XGBoost, LightGBM & CatBoost

XGBoost, LightGBM, and CatBoost are production-grade versions of gradient-boosted decision trees: they build an additive model one small tree at a time, use derivatives of the loss to decide useful corrections, and add regularization or validation checks so a flattering training score is not mistaken for a durable model.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build the boosting ideas one piece at a time. Run each cell in order and read the printed intermediate values — every calculation is small enough to inspect, and the walkthrough uses a `_w` suffix so it never clashes with the examples below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

### 1. Empirical risk and additive corrections

Boosting begins with an empirical-risk question: how bad are the current predictions on the rows we have? Instead of replacing the whole model, a boosted tree adds a correction. For squared loss, the negative gradient is just the residual `y - pred`, so the next tree tries to explain what the current model still misses.

In [ ]:
x_w = np.array([0., 1., 2., 3., 4., 5.])
y_w = np.array([1.0, 1.4, 1.8, 3.1, 3.6, 4.0])
pred0_w = np.full_like(y_w, y_w.mean())
losses_w = 0.5 * (pred0_w - y_w) ** 2
R_S_w = float(np.mean([0.180, 0.096, 0.420]))
print("initial constant:", round(float(pred0_w[0]), 3))
print("toy empirical risk from lesson:", round(R_S_w, 3))
assert round(R_S_w, 3) == 0.232

▶ What you'll see: the model starts as one constant prediction, and the lesson's checked average loss is 0.232.

In [ ]:
resid_w = y_w - pred0_w
left_w = x_w <= 2
right_w = ~left_w
leaf_values_w = np.array([resid_w[left_w].mean(), resid_w[right_w].mean()])
correction_w = np.where(left_w, leaf_values_w[0], leaf_values_w[1])
pred1_w = pred0_w + correction_w
print("residuals:", np.round(resid_w, 3))
print("leaf corrections:", np.round(leaf_values_w, 3))
print("MSE before -> after:", round(np.mean((y_w - pred0_w)**2), 3), "->", round(np.mean((y_w - pred1_w)**2), 3))
assert np.mean((y_w - pred1_w)**2) < np.mean((y_w - pred0_w)**2)

▶ What you'll see: one two-leaf tree explains low-x negative residuals and high-x positive residuals, reducing error.

In [ ]:
plt.figure(figsize=(5, 3))
plt.scatter(x_w, y_w, color="black", label="data")
plt.step(x_w, pred0_w, where="mid", color="gray", label="constant")
plt.step(x_w, pred1_w, where="mid", color="seagreen", label="after one tree")
plt.title("1: boosting adds a residual correction")
plt.xlabel("x")
plt.ylabel("prediction")
plt.legend()
plt.show()

▶ What you'll see: the green step function moves the flat gray model toward the observed points.

*Why it's done this way:* empirical risk averages per-row losses so model selection has a single scale. Boosting then follows the steepest local improvement: for squared loss, the derivative of `0.5(pred-y)^2` is `pred-y`, so the negative gradient is the residual. A small tree is a constrained correction, which lowers training loss without giving every row its own free parameter.

### 2. XGBoost's second-order leaf score

XGBoost makes the correction decision with a second-order Taylor approximation. For one leaf with gradients `G=sum(g_i)` and Hessians `H=sum(h_i)`, the best constant leaf value is `w* = -G/(H+lambda)`, and the approximate gain is `0.5*G^2/(H+lambda) - gamma`. That is the formula behind the lesson's `g_i f_i + 1/2 h_i f_i^2 + Omega(f)` expression.

In [ ]:
g_w = pred0_w - y_w
h_w = np.ones_like(g_w)
lam_w = 1.0
gamma_w = 0.05
G_w, H_w = float(g_w.sum()), float(h_w.sum())
w_star_w = -G_w / (H_w + lam_w)
leaf_score_w = -0.5 * G_w**2 / (H_w + lam_w) + gamma_w
print("G, H:", round(G_w, 3), H_w)
print("best one-leaf weight:", round(w_star_w, 3))
print("regularized objective contribution:", round(leaf_score_w, 3))
assert abs(round(w_star_w, 3)) == 0.0

▶ What you'll see: the constant model has zero total gradient, so one more unsplit leaf would not help.

In [ ]:
G_L_w, H_L_w = float(g_w[left_w].sum()), float(h_w[left_w].sum())
G_R_w, H_R_w = float(g_w[right_w].sum()), float(h_w[right_w].sum())
gain_w = 0.5 * (G_L_w**2/(H_L_w+lam_w) + G_R_w**2/(H_R_w+lam_w) - G_w**2/(H_w+lam_w)) - gamma_w
wL_w = -G_L_w / (H_L_w + lam_w)
wR_w = -G_R_w / (H_R_w + lam_w)
print("left/right weights:", round(wL_w, 3), round(wR_w, 3))
print("split gain:", round(gain_w, 3))
assert gain_w > 0

▶ What you'll see: the split has positive gain because two leaves can move in opposite directions.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["left G", "right G", "gain"], [G_L_w, G_R_w, gain_w], color=["steelblue", "darkorange", "seagreen"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("2: second-order split score")
plt.show()

▶ What you'll see: opposite-signed gradient sums create a useful split, while the gain bar shows the penalty-adjusted reward.

*Why it's done this way:* the Taylor expansion turns an arbitrary differentiable loss into a local quadratic problem. Quadratics have closed-form optima, so XGBoost can score many candidate tree splits quickly. `lambda` shrinks leaf values by increasing the denominator, and `gamma` charges each split, so raw fit must overcome complexity cost before the tree grows.

### 3. LightGBM-style histogram split search

Exact split search sorts every distinct feature value, which can be expensive. LightGBM bins continuous features into histograms, accumulates gradient and Hessian sums per bin, and scores splits between bins. The approximation is fast because the tree searches a small number of bins instead of every raw value.

In [ ]:
x2_w = np.array([0.2, 0.4, 0.9, 1.2, 1.8, 2.2, 2.7, 3.1, 3.7, 4.0])
y2_w = np.array([1.0, 1.1, 1.3, 1.5, 2.0, 2.7, 3.0, 3.4, 3.8, 4.2])
pred2_w = np.full_like(y2_w, y2_w.mean())
g2_w = pred2_w - y2_w
h2_w = np.ones_like(y2_w)
bins_w = np.digitize(x2_w, np.array([1.0, 2.5, 3.5]))
print("bin ids:", bins_w)
print("gradient sums by bin:", [round(float(g2_w[bins_w == b].sum()), 3) for b in range(4)])
assert len(np.unique(bins_w)) == 4

▶ What you'll see: ten raw values are compressed into four gradient/Hessian buckets.

In [ ]:
G_bins_w = np.array([g2_w[bins_w == b].sum() for b in range(4)])
H_bins_w = np.array([h2_w[bins_w == b].sum() for b in range(4)])
Gtot_w, Htot_w = G_bins_w.sum(), H_bins_w.sum()
gains_w = []
for cut_w in range(3):
    GL_w, HL_w = G_bins_w[:cut_w+1].sum(), H_bins_w[:cut_w+1].sum()
    GR_w, HR_w = Gtot_w - GL_w, Htot_w - HL_w
    gains_w.append(0.5 * (GL_w**2/(HL_w+1) + GR_w**2/(HR_w+1) - Gtot_w**2/(Htot_w+1)))
best_cut_w = int(np.argmax(gains_w))
print("histogram gains:", np.round(gains_w, 3))
print("best bin cut:", best_cut_w)
assert best_cut_w == 1

▶ What you'll see: the best split is found by scanning three bin boundaries rather than nine raw gaps.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(range(4), G_bins_w, color="slateblue")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("3: LightGBM histogram gradient sums")
plt.xlabel("feature bin")
plt.ylabel("sum of gradients")
plt.show()

▶ What you'll see: early bins have positive gradients and later bins have negative gradients, so a split separates two correction directions.

*Why it's done this way:* split gain depends only on sums of gradients and Hessians, not on individual rows after they are assigned to a candidate side. Binning preserves the approximate order structure while making the search cache-friendly and much cheaper. The mathematical compromise is bias for speed: fewer split locations, but nearly the same gain calculation.

### 4. CatBoost-style categorical statistics without leakage

Trees do not naturally know that categories like `red`, `blue`, and `green` can be ordered by historical target behavior. CatBoost's key idea is to encode categories with target statistics while avoiding leakage: for a row, use only earlier rows (or ordered permutations) to compute the category mean, smoothed toward a prior.

In [ ]:
cat_w = np.array(["red", "blue", "red", "green", "blue", "red", "green", "blue"])
ycat_w = np.array([1., 0., 1., 0., 1., 1., 0., 1.])
prior_w = float(ycat_w.mean())
alpha_w = 2.0
naive_red_w = ycat_w[cat_w == "red"].mean()
print("global prior:", round(prior_w, 3))
print("naive red mean:", round(float(naive_red_w), 3))
assert round(prior_w, 3) == 0.625

▶ What you'll see: the naive category mean uses the current row's target, which would leak the answer.

In [ ]:
ordered_w = []
for i_w in range(len(cat_w)):
    past_w = (cat_w[:i_w] == cat_w[i_w])
    count_w = int(past_w.sum())
    total_w = float(ycat_w[:i_w][past_w].sum())
    ordered_w.append((total_w + alpha_w * prior_w) / (count_w + alpha_w))
ordered_w = np.array(ordered_w)
print("ordered encodings:", np.round(ordered_w, 3))
print("row 0 red uses prior only:", round(float(ordered_w[0]), 3))
assert round(float(ordered_w[0]), 3) == 0.625

▶ What you'll see: the first occurrence of each category falls back to the prior; later rows use only past evidence.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(ordered_w, marker="o", color="darkorange")
plt.axhline(prior_w, color="gray", linestyle="--", label="prior")
plt.title("4: ordered target encoding")
plt.xlabel("row order")
plt.ylabel("encoded category value")
plt.legend()
plt.show()

▶ What you'll see: encodings start at the prior and move only when previous same-category labels exist.

*Why it's done this way:* target statistics convert a category into a numeric signal a tree can split on, but using the row's own target would make training look artificially good. Ordered encoding estimates the conditional mean from information that would have been available before the row, and smoothing prevents rare categories from receiving extreme values after one example.

### 5. Cost-aware selection, validation gaps, and stability

The lesson's selection arithmetic says raw training fit is not enough. We compare the empirical term, the complexity cost, a tempting alternative, and a stabilizing setting. In real boosting libraries these are knobs such as shrinkage, tree depth, leaf penalties, subsampling, early stopping, and categorical smoothing.

In [ ]:
raw_losses_w = np.array([0.180, 0.096, 0.420])
cost_w = 0.080
score_w = raw_losses_w.mean() + cost_w
alternative_w = 0.356
gap_w = alternative_w - score_w
rel_gap_w = gap_w / alternative_w
stable_w = 0.80 * score_w
print("raw mean:", round(float(raw_losses_w.mean()), 3))
print("score with cost:", round(score_w, 3))
print("gap and relative gap:", round(gap_w, 3), round(rel_gap_w, 3))
print("stabilized score:", round(stable_w, 3))
assert round(score_w, 3) == 0.312
assert round(gap_w, 3) == 0.044

▶ What you'll see: the full decision score is 0.312, not the raw 0.232, and a stabilizing knob lowers it to about 0.250.

In [ ]:
train_curve_w = np.array([0.50, 0.38, 0.30, 0.24, 0.20, 0.17])
val_curve_w = np.array([0.52, 0.40, 0.33, 0.31, 0.32, 0.35])
best_round_w = int(np.argmin(val_curve_w)) + 1
print("best validation round:", best_round_w)
print("train keeps falling:", train_curve_w[-1] < train_curve_w[best_round_w-1])
assert best_round_w == 4

▶ What you'll see: training loss keeps improving after round 4, but validation loss starts getting worse.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(np.arange(1, 7), train_curve_w, marker="o", label="train")
plt.plot(np.arange(1, 7), val_curve_w, marker="o", label="validation")
plt.axvline(best_round_w, color="red", linestyle="--", label="early stop")
plt.title("5: validation selects the durable model")
plt.xlabel("boosting round")
plt.ylabel("loss")
plt.legend()
plt.show()

▶ What you'll see: the validation curve bottoms out before the training curve, making early stopping a stability rule.

*Why it's done this way:* flexible trees can always chase training fragments, so model choice must include a cost or an unseen-data check. The gap measures how much evidence separates alternatives; if the gap is small, sampling noise can flip the decision. Stabilizing knobs intentionally sacrifice some apparent flexibility to reduce variance and improve future performance.

## 🛠️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

## 🟢 Basics (warm-up)

### Basic 1 — Average empirical loss

**Goal.** Average three per-example losses, because boosted trees optimize empirical risk before any cost term is added.

In [ ]:
losses_b1 = np.array([0.180, 0.096, 0.420])
risk_b1 = float(losses_b1.mean())
print("empirical risk:", round(risk_b1, 3))
assert round(risk_b1, 3) == 0.232
plt.figure(figsize=(4, 3))
plt.bar(["row1", "row2", "row3"], losses_b1, color="steelblue")
plt.axhline(risk_b1, color="red", linestyle="--", label="mean")
plt.title("Basic 1: per-row losses")
plt.legend()
plt.show()

▶ What you'll see: the dashed line is the average loss each model tries to reduce.

👀 Takeaway: empirical risk is the mean of the row-level losses, not the best or worst row alone.

### Basic 2 — Add the complexity cost

**Goal.** Add a penalty to raw fit, because XGBoost-style selection compares the full regularized score.

In [ ]:
risk_b2 = 0.232
cost_b2 = 0.080
score_b2 = risk_b2 + cost_b2
print("risk + cost =", round(score_b2, 3))
assert round(score_b2, 3) == 0.312
plt.figure(figsize=(4, 3))
plt.bar(["raw risk", "cost", "total"], [risk_b2, cost_b2, score_b2], color=["gray", "orange", "seagreen"])
plt.title("Basic 2: full selection score")
plt.show()

▶ What you'll see: the total score is visibly larger than the raw risk.

👀 Takeaway: regularized boosting chooses trees by fit after paying for complexity.

### Basic 3 — Compute residual gradients

**Goal.** Derive the negative gradient for squared loss, because the next tree fits what the current model misses.

In [ ]:
y_b3 = np.array([1.0, 1.4, 1.8, 3.1, 3.6, 4.0])
pred_b3 = np.full_like(y_b3, y_b3.mean())
grad_b3 = pred_b3 - y_b3
neg_grad_b3 = -grad_b3
print("negative gradients:", np.round(neg_grad_b3, 3))
assert round(float(neg_grad_b3.mean()), 6) == 0.0
plt.figure(figsize=(4, 3))
plt.bar(range(len(y_b3)), neg_grad_b3, color="purple")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 3: residuals as negative gradients")
plt.show()

▶ What you'll see: rows below the mean have negative corrections, rows above the mean have positive corrections.

👀 Takeaway: for squared loss, gradient boosting is residual fitting.

### Basic 4 — Fit a two-leaf correction

**Goal.** Average residuals inside two leaves, because a decision tree correction assigns one value per leaf.

In [ ]:
x_b4 = np.array([0., 1., 2., 3., 4., 5.])
y_b4 = np.array([1.0, 1.4, 1.8, 3.1, 3.6, 4.0])
pred_b4 = np.full_like(y_b4, y_b4.mean())
res_b4 = y_b4 - pred_b4
left_b4 = x_b4 <= 2
leaf_b4 = np.array([res_b4[left_b4].mean(), res_b4[~left_b4].mean()])
print("leaf residual means:", np.round(leaf_b4, 3))
assert np.allclose(np.round(leaf_b4, 3), [-1.083, 1.083])
plt.figure(figsize=(4, 3))
plt.bar(["left leaf", "right leaf"], leaf_b4, color=["steelblue", "darkorange"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 4: two leaf corrections")
plt.show()

▶ What you'll see: the split learns equal-and-opposite corrections for the two regions.

👀 Takeaway: a tree turns many row residuals into a few shared correction values.

### Basic 5 — Apply a learning rate

**Goal.** Shrink the tree update, because boosting usually takes cautious steps rather than adding the full correction.

In [ ]:
pred_b5 = np.full(6, 2.4833333333)
correction_b5 = np.array([-1.0833333333, -1.0833333333, -1.0833333333, 1.0833333333, 1.0833333333, 1.0833333333])
eta_b5 = 0.3
updated_b5 = pred_b5 + eta_b5 * correction_b5
print("updated predictions:", np.round(updated_b5, 3))
assert round(float(updated_b5[0]), 3) == 2.158
plt.figure(figsize=(4, 3))
plt.step(range(6), pred_b5, where="mid", label="before")
plt.step(range(6), updated_b5, where="mid", label="after shrinkage")
plt.title("Basic 5: learning-rate shrinkage")
plt.legend()
plt.show()

▶ What you'll see: predictions move in the right direction, but only 30% of the full leaf correction is applied.

👀 Takeaway: learning rate trades speed for stability.

### Basic 6 — Compute an XGBoost leaf weight

**Goal.** Use `-G/(H+lambda)`, because the best regularized leaf value has a closed form under the quadratic approximation.

In [ ]:
g_b6 = np.array([1.2, 0.8, 1.0])
h_b6 = np.ones(3)
lam_b6 = 1.0
G_b6 = float(g_b6.sum())
H_b6 = float(h_b6.sum())
w_b6 = -G_b6 / (H_b6 + lam_b6)
print("leaf weight:", round(w_b6, 3))
assert round(w_b6, 3) == -0.75
plt.figure(figsize=(4, 3))
plt.bar(["G", "H+lambda", "w*"], [G_b6, H_b6 + lam_b6, w_b6], color=["gray", "orange", "seagreen"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 6: regularized leaf value")
plt.show()

▶ What you'll see: a positive gradient sum produces a negative correction.

👀 Takeaway: the sign of the gradient sum tells the leaf which way to move predictions.

### Basic 7 — Score a split gain

**Goal.** Compare parent and child leaf scores, because a split is useful only if children improve the regularized objective enough.

In [ ]:
GL_b7, HL_b7 = 3.0, 3.0
GR_b7, HR_b7 = -3.0, 3.0
G_b7, H_b7 = GL_b7 + GR_b7, HL_b7 + HR_b7
lam_b7 = 1.0
gamma_b7 = 0.05
gain_b7 = 0.5 * (GL_b7**2/(HL_b7+lam_b7) + GR_b7**2/(HR_b7+lam_b7) - G_b7**2/(H_b7+lam_b7)) - gamma_b7
print("split gain:", round(gain_b7, 3))
assert round(gain_b7, 3) == 2.2
plt.figure(figsize=(4, 3))
plt.bar(["left score", "right score", "gain"], [GL_b7**2/(HL_b7+lam_b7), GR_b7**2/(HR_b7+lam_b7), gain_b7], color="teal")
plt.title("Basic 7: gain after split cost")
plt.show()

▶ What you'll see: two child leaves with opposite gradients create a positive gain after paying `gamma`.

👀 Takeaway: XGBoost grows a tree when the gain justifies the extra leaf complexity.

### Basic 8 — Bin a feature into histograms

**Goal.** Replace raw values by bins, because LightGBM split search accumulates derivative sums in histograms.

In [ ]:
x_b8 = np.array([0.2, 0.4, 0.9, 1.2, 1.8, 2.2, 2.7, 3.1])
g_b8 = np.array([1.2, 1.0, 0.8, 0.3, 0.1, -0.4, -0.8, -1.1])
bins_b8 = np.digitize(x_b8, np.array([1.0, 2.0, 3.0]))
G_bins_b8 = np.array([g_b8[bins_b8 == b].sum() for b in range(4)])
print("gradient per bin:", np.round(G_bins_b8, 3))
assert round(float(G_bins_b8.sum()), 3) == round(float(g_b8.sum()), 3)
plt.figure(figsize=(4, 3))
plt.bar(range(4), G_bins_b8, color="slateblue")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 8: histogram gradient sums")
plt.xlabel("bin")
plt.show()

▶ What you'll see: derivative information is preserved as bin totals.

👀 Takeaway: histogram boosting speeds split search by aggregating sufficient statistics.

### Basic 9 — Smooth a category mean

**Goal.** Blend a category's observed rate with the global prior, because rare categories need protection from noisy extremes.

In [ ]:
y_b9 = np.array([1., 0., 1., 1., 0., 1.])
cat_b9 = np.array(["a", "a", "b", "b", "b", "c"])
prior_b9 = y_b9.mean()
alpha_b9 = 2.0
mask_b9 = cat_b9 == "c"
encoded_c_b9 = (y_b9[mask_b9].sum() + alpha_b9 * prior_b9) / (mask_b9.sum() + alpha_b9)
print("prior:", round(float(prior_b9), 3), "smoothed c:", round(float(encoded_c_b9), 3))
assert round(float(encoded_c_b9), 3) == 0.778
plt.figure(figsize=(4, 3))
plt.bar(["prior", "raw c", "smoothed c"], [prior_b9, y_b9[mask_b9].mean(), encoded_c_b9], color=["gray", "red", "seagreen"])
plt.title("Basic 9: smoothed category statistic")
plt.show()

▶ What you'll see: the smoothed value is less extreme than the raw one-example category mean.

👀 Takeaway: CatBoost-style category signals are target means with leakage and rarity controls.

### Basic 10 — Choose the lower full score

**Goal.** Compare baseline, flexible, and stabilized scores, because the best model is chosen on the full decision scale.

In [ ]:
scores_b10 = np.array([0.312, 0.356, 0.250])
labels_b10 = np.array(["baseline", "flexible", "stabilized"])
best_b10 = labels_b10[int(np.argmin(scores_b10))]
print("best model:", best_b10, "score:", scores_b10.min())
assert best_b10 == "stabilized"
plt.figure(figsize=(4, 3))
plt.bar(labels_b10, scores_b10, color=["gray", "orange", "seagreen"])
plt.title("Basic 10: lower regularized score wins")
plt.ylabel("decision score")
plt.xticks(rotation=12)
plt.show()

▶ What you'll see: the stabilized model has the lowest bar.

👀 Takeaway: selection uses the complete score, not the prettiest raw training fragment.

## 🟡 Easy

### Easy 1 — Build a tiny gradient-boosted stump model

**Goal.** Train several residual-fitting stumps, because boosting is an additive sequence of weak trees.

In [ ]:
x_e1 = np.linspace(0, 5, 12)
y_e1 = 1 + 0.7 * x_e1 + np.where(x_e1 > 2.5, 0.8, -0.4)
pred_e1 = np.full_like(y_e1, y_e1.mean())
eta_e1 = 0.4
cuts_e1 = np.array([1.5, 2.5, 3.5])
train_mse_e1 = []
for m_e1 in range(5):
    residual_e1 = y_e1 - pred_e1
    best_loss_e1, best_corr_e1 = 1e9, None
    for cut_e1 in cuts_e1:
        left_e1 = x_e1 <= cut_e1
        vals_e1 = np.array([residual_e1[left_e1].mean(), residual_e1[~left_e1].mean()])
        corr_e1 = np.where(left_e1, vals_e1[0], vals_e1[1])
        loss_e1 = np.mean((residual_e1 - corr_e1)**2)
        if loss_e1 < best_loss_e1:
            best_loss_e1, best_corr_e1 = loss_e1, corr_e1
    pred_e1 = pred_e1 + eta_e1 * best_corr_e1
    train_mse_e1.append(np.mean((y_e1 - pred_e1)**2))
print("MSE curve:", np.round(train_mse_e1, 3))
assert train_mse_e1[-1] < train_mse_e1[0]
plt.figure(figsize=(5, 3))
plt.plot(train_mse_e1, marker="o", color="teal")
plt.title("Easy 1: additive stump training")
plt.xlabel("boosting round")
plt.ylabel("MSE")
plt.show()

▶ What you'll see: training MSE drops as each stump explains remaining residual structure.

👀 Takeaway: gradient boosting is repeated small corrective steps, not one large tree.

### Easy 2 — Implement second-order split search

**Goal.** Scan candidate splits with XGBoost's gain formula, because the best split maximizes regularized quadratic improvement.

In [ ]:
x_e2 = np.array([0., 1., 2., 3., 4., 5.])
y_e2 = np.array([1.0, 1.4, 1.8, 3.1, 3.6, 4.0])
pred_e2 = np.full_like(y_e2, y_e2.mean())
g_e2 = pred_e2 - y_e2
h_e2 = np.ones_like(y_e2)
lam_e2, gamma_e2 = 1.0, 0.05
gains_e2 = []
for cut_e2 in [0.5, 1.5, 2.5, 3.5, 4.5]:
    left_e2 = x_e2 <= cut_e2
    GL_e2, HL_e2 = g_e2[left_e2].sum(), h_e2[left_e2].sum()
    GR_e2, HR_e2 = g_e2[~left_e2].sum(), h_e2[~left_e2].sum()
    G_e2, H_e2 = g_e2.sum(), h_e2.sum()
    gains_e2.append(0.5*(GL_e2**2/(HL_e2+lam_e2)+GR_e2**2/(HR_e2+lam_e2)-G_e2**2/(H_e2+lam_e2))-gamma_e2)
best_idx_e2 = int(np.argmax(gains_e2))
print("gains:", np.round(gains_e2, 3), "best cut:", [0.5, 1.5, 2.5, 3.5, 4.5][best_idx_e2])
assert [0.5, 1.5, 2.5, 3.5, 4.5][best_idx_e2] == 2.5
plt.figure(figsize=(5, 3))
plt.plot([0.5, 1.5, 2.5, 3.5, 4.5], gains_e2, marker="o", color="purple")
plt.title("Easy 2: XGBoost gain scan")
plt.xlabel("candidate cut")
plt.ylabel("gain")
plt.show()

▶ What you'll see: the best gain occurs at the split separating the low and high response regions.

👀 Takeaway: second-order split search is just a disciplined score for possible tree branches.

### Easy 3 — Compare exact and histogram gains

**Goal.** Approximate exact split search with bins, because LightGBM trades split precision for speed.

In [ ]:
x_e3 = np.array([0.2, 0.4, 0.9, 1.2, 1.8, 2.2, 2.7, 3.1, 3.7, 4.0])
y_e3 = np.array([1.0, 1.1, 1.3, 1.5, 2.0, 2.7, 3.0, 3.4, 3.8, 4.2])
g_e3 = y_e3.mean() - y_e3
h_e3 = np.ones_like(y_e3)
def gain_e3(left_e3):
    GL_e3, HL_e3 = g_e3[left_e3].sum(), h_e3[left_e3].sum()
    GR_e3, HR_e3 = g_e3[~left_e3].sum(), h_e3[~left_e3].sum()
    return 0.5*(GL_e3**2/(HL_e3+1)+GR_e3**2/(HR_e3+1)-g_e3.sum()**2/(h_e3.sum()+1))
exact_cuts_e3 = (x_e3[:-1] + x_e3[1:]) / 2
exact_gains_e3 = np.array([gain_e3(x_e3 <= c_e3) for c_e3 in exact_cuts_e3])
bin_edges_e3 = np.array([1.0, 2.5, 3.5])
hist_cuts_e3 = np.array([1.0, 2.5, 3.5])
hist_gains_e3 = np.array([gain_e3(x_e3 <= c_e3) for c_e3 in hist_cuts_e3])
print("best exact cut:", round(float(exact_cuts_e3[np.argmax(exact_gains_e3)]), 2))
print("best histogram cut:", round(float(hist_cuts_e3[np.argmax(hist_gains_e3)]), 2))
assert abs(hist_cuts_e3[np.argmax(hist_gains_e3)] - exact_cuts_e3[np.argmax(exact_gains_e3)]) <= 0.6
plt.figure(figsize=(5, 3))
plt.plot(exact_cuts_e3, exact_gains_e3, marker="o", label="exact")
plt.scatter(hist_cuts_e3, hist_gains_e3, color="red", label="hist")
plt.title("Easy 3: exact vs binned split gains")
plt.xlabel("cut")
plt.ylabel("gain")
plt.legend()
plt.show()

▶ What you'll see: histogram candidates are fewer but locate a similar high-gain region.

👀 Takeaway: LightGBM accelerates boosting by searching useful bins instead of every raw gap.

### Easy 4 — Ordered target encoding for a category

**Goal.** Encode categories using only earlier rows, because CatBoost avoids target leakage from the row being predicted.

In [ ]:
cat_e4 = np.array(["red", "blue", "red", "green", "blue", "red", "green", "blue"])
y_e4 = np.array([1., 0., 1., 0., 1., 1., 0., 1.])
prior_e4 = y_e4.mean()
alpha_e4 = 2.0
enc_e4 = []
for i_e4 in range(len(cat_e4)):
    past_e4 = cat_e4[:i_e4] == cat_e4[i_e4]
    enc_e4.append((y_e4[:i_e4][past_e4].sum() + alpha_e4 * prior_e4) / (past_e4.sum() + alpha_e4))
enc_e4 = np.array(enc_e4)
print("ordered encodings:", np.round(enc_e4, 3))
assert round(float(enc_e4[0]), 3) == round(float(prior_e4), 3)
plt.figure(figsize=(5, 3))
plt.plot(enc_e4, marker="o", color="darkorange")
plt.axhline(prior_e4, color="gray", linestyle="--")
plt.title("Easy 4: leakage-safe category encoding")
plt.xlabel("row")
plt.ylabel("encoding")
plt.show()

▶ What you'll see: early category occurrences use the prior, while later ones reflect past same-category outcomes.

👀 Takeaway: category encodings must be computed as if the current label is unknown.

### Easy 5 — Early stopping on validation loss

**Goal.** Pick the best boosting round from validation loss, because training loss alone can favor overfitting.

In [ ]:
train_e5 = np.array([0.50, 0.38, 0.30, 0.24, 0.20, 0.17, 0.15])
val_e5 = np.array([0.52, 0.40, 0.33, 0.31, 0.32, 0.35, 0.38])
best_round_e5 = int(np.argmin(val_e5)) + 1
print("best validation round:", best_round_e5, "validation loss:", val_e5.min())
assert best_round_e5 == 4
plt.figure(figsize=(5, 3))
plt.plot(np.arange(1, 8), train_e5, marker="o", label="train")
plt.plot(np.arange(1, 8), val_e5, marker="o", label="validation")
plt.axvline(best_round_e5, color="red", linestyle="--")
plt.title("Easy 5: early stopping")
plt.xlabel("round")
plt.ylabel("loss")
plt.legend()
plt.show()

▶ What you'll see: validation loss bottoms out even though training loss continues downward.

👀 Takeaway: boosted trees should be selected by validation behavior, not by endless training improvement.

## 🔴 Advanced

### Advanced 1 — Tune tree complexity with a gamma penalty

**Goal.** Sweep split penalties, because `gamma` decides how much gain a split must earn before it is accepted.

In [ ]:
base_gain_a1 = 2.25
gammas_a1 = np.array([0.0, 0.5, 1.0, 2.0, 2.5])
net_gain_a1 = base_gain_a1 - gammas_a1
accepted_a1 = net_gain_a1 > 0
print("net gains:", np.round(net_gain_a1, 3))
print("accepted:", accepted_a1.astype(int))
assert accepted_a1[-1] == False
plt.figure(figsize=(5, 3))
plt.plot(gammas_a1, net_gain_a1, marker="o", color="crimson")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Advanced 1: gamma prunes weak splits")
plt.xlabel("gamma")
plt.ylabel("net gain")
plt.show()

▶ What you'll see: increasing `gamma` pushes marginal splits below zero net gain.

👀 Takeaway: split penalties turn raw improvement into a cost-aware tree-growing decision.

### Advanced 2 — Sweep learning rates and rounds

**Goal.** Compare shrinkage schedules, because smaller learning rates usually need more rounds but can generalize more smoothly.

In [ ]:
x_a2 = np.linspace(0, 5, 20)
y_a2 = np.sin(x_a2) + 0.5 * x_a2
cuts_a2 = np.linspace(0.5, 4.5, 5)
rates_a2 = np.array([0.1, 0.3, 0.7])
final_mse_a2 = []
for eta_a2 in rates_a2:
    pred_a2 = np.full_like(y_a2, y_a2.mean())
    for round_a2 in range(8):
        res_a2 = y_a2 - pred_a2
        best_loss_a2, best_corr_a2 = 1e9, None
        for cut_a2 in cuts_a2:
            left_a2 = x_a2 <= cut_a2
            vals_a2 = np.array([res_a2[left_a2].mean(), res_a2[~left_a2].mean()])
            corr_a2 = np.where(left_a2, vals_a2[0], vals_a2[1])
            loss_a2 = np.mean((res_a2 - corr_a2)**2)
            if loss_a2 < best_loss_a2:
                best_loss_a2, best_corr_a2 = loss_a2, corr_a2
        pred_a2 += eta_a2 * best_corr_a2
    final_mse_a2.append(np.mean((y_a2 - pred_a2)**2))
print("final MSE by eta:", np.round(final_mse_a2, 4))
assert min(final_mse_a2) < max(final_mse_a2)
plt.figure(figsize=(5, 3))
plt.plot(rates_a2, final_mse_a2, marker="o", color="navy")
plt.title("Advanced 2: shrinkage sweep")
plt.xlabel("learning rate")
plt.ylabel("training MSE after 8 rounds")
plt.show()

▶ What you'll see: different learning rates reach different errors after the same number of rounds.

👀 Takeaway: learning rate and number of trees are coupled hyperparameters.

### Advanced 3 — Compare level-wise and leaf-wise growth

**Goal.** Contrast split allocation strategies, because LightGBM often grows the leaf with largest gain rather than expanding every level evenly.

In [ ]:
leaf_gains_a3 = np.array([0.9, 0.2, 0.7, 0.1])
levelwise_gain_a3 = leaf_gains_a3[:2].sum()
leafwise_gain_a3 = leaf_gains_a3[np.argsort(leaf_gains_a3)[-2:]].sum()
print("level-wise two splits gain:", round(float(levelwise_gain_a3), 3))
print("leaf-wise top two splits gain:", round(float(leafwise_gain_a3), 3))
assert leafwise_gain_a3 > levelwise_gain_a3
plt.figure(figsize=(5, 3))
plt.bar(["level-wise", "leaf-wise"], [levelwise_gain_a3, leafwise_gain_a3], color=["gray", "seagreen"])
plt.title("Advanced 3: choosing where to grow")
plt.ylabel("total selected gain")
plt.show()

▶ What you'll see: spending splits on the highest-gain leaves gives more immediate objective improvement.

👀 Takeaway: leaf-wise growth can be efficient but needs depth or leaf-count controls to avoid overfitting.

### Advanced 4 — Quantify category leakage

**Goal.** Compare naive and ordered category encodings on the training labels, because leakage makes categories look more predictive than they really are.

In [ ]:
cat_a4 = np.array(["a", "a", "a", "b", "b", "c", "c", "c"])
y_a4 = np.array([1., 0., 1., 0., 0., 1., 1., 0.])
prior_a4 = y_a4.mean()
naive_a4 = np.array([y_a4[cat_a4 == c_a4].mean() for c_a4 in cat_a4])
ordered_a4 = []
for i_a4 in range(len(cat_a4)):
    past_a4 = cat_a4[:i_a4] == cat_a4[i_a4]
    ordered_a4.append((y_a4[:i_a4][past_a4].sum() + 2 * prior_a4) / (past_a4.sum() + 2))
ordered_a4 = np.array(ordered_a4)
naive_mse_a4 = np.mean((y_a4 - naive_a4)**2)
ordered_mse_a4 = np.mean((y_a4 - ordered_a4)**2)
print("naive MSE:", round(float(naive_mse_a4), 3), "ordered MSE:", round(float(ordered_mse_a4), 3))
assert naive_mse_a4 < ordered_mse_a4
plt.figure(figsize=(5, 3))
plt.bar(["naive leaked", "ordered"], [naive_mse_a4, ordered_mse_a4], color=["red", "seagreen"])
plt.title("Advanced 4: leakage looks too good")
plt.ylabel("training MSE")
plt.show()

▶ What you'll see: the naive encoding has lower training error because it used label information too directly.

👀 Takeaway: a too-good category statistic can be a leakage symptom, not a better model.

### Advanced 5 — Validate cost-aware model selection

**Goal.** Combine raw risk, cost, and validation uncertainty, because the winning boosted model should survive the full decision rule.

In [ ]:
names_a5 = np.array(["baseline", "flexible", "stabilized"])
raw_a5 = np.array([0.232, 0.266, 0.220])
cost_a5 = np.array([0.080, 0.090, 0.030])
val_noise_a5 = np.array([0.020, 0.035, 0.018])
score_a5 = raw_a5 + cost_a5 + val_noise_a5
winner_a5 = names_a5[int(np.argmin(score_a5))]
print("decision scores:", dict(zip(names_a5, np.round(score_a5, 3))))
print("winner:", winner_a5)
assert winner_a5 == "stabilized"
plt.figure(figsize=(5, 3))
plt.bar(names_a5, score_a5, color=["gray", "orange", "seagreen"])
plt.title("Advanced 5: full decision score")
plt.ylabel("risk + cost + uncertainty")
plt.xticks(rotation=12)
plt.show()

▶ What you'll see: the flexible model is not selected once cost and uncertainty are included.

👀 Takeaway: production boosting decisions should include fit, complexity, and validation stability on the same scale.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Modern boosting libraries make gradient boosting fast, regularized, and production-friendly.

XGBoost, LightGBM, and CatBoost are production-grade gradient boosting systems. To keep this rebuild CPU-only with no installs, the notebook uses sklearn's HistGradientBoostingClassifier as the stand-in while preserving the same second-order, regularized boosting story. Save a copy to Drive to edit.

In [ ]:

import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
np.random.seed(7)

def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...].

    All X are 2-D float feature matrices, y integer labels, so one classifier runs unchanged
    across every rung (the 'watch it scale' story). Rungs get harder: clean+separable -> real
    high-dimensional. D1 is hand-built and fully inspectable.
    """
    rungs = []

    # D1 — four hand-placed 2-D points, 2 classes, clearly separable.
    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    # D2 — clean, well-separated Gaussian blobs.
    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    # D3 — non-linear, overlapping two-moons with noise.
    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    # D4 — real: Wine, 13 features, 3 classes.
    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    # D5 — real, harder: Breast Cancer, 30 features, class imbalance.
    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)



def _fit_predict(model, x_tr, y_tr, x_te):
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def _stump_adaboost(n_estimators=60, learning_rate=0.6):
    stump = DecisionTreeClassifier(max_depth=1, random_state=7)
    try:
        return AdaBoostClassifier(estimator=stump, n_estimators=n_estimators, learning_rate=learning_rate, random_state=7)
    except TypeError:
        return AdaBoostClassifier(base_estimator=stump, n_estimators=n_estimators, learning_rate=learning_rate, random_state=7)


def _project2d(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 1:
        return np.c_[X[:, 0], np.zeros(X.shape[0])]
    return X[:, :2]


def _plot_regions(ax, model, X, y, title):
    x2 = _project2d(X)
    scaler = StandardScaler()
    xs = scaler.fit_transform(x2)
    fitted = clone(model)
    try:
        fitted.fit(xs, y)
    except ValueError:
        fitted = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
        fitted.fit(xs, y)
    x_min = xs[:, 0].min() - 0.8
    x_max = xs[:, 0].max() + 0.8
    y_min = xs[:, 1].min() - 0.8
    y_max = xs[:, 1].max() + 0.8
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 80), np.linspace(y_min, y_max, 80))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = fitted.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, cmap="tab10")
    ax.scatter(xs[:, 0], xs[:, 1], c=y, s=16, cmap="tab10", edgecolor="k", linewidth=0.2)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


def summarize_ladder(rungs):
    for name, X, y in rungs:
        classes, counts = np.unique(y, return_counts=True)
        print(f"{name}: X={X.shape}, classes={dict(zip(classes.tolist(), counts.tolist()))}")
    sample_name, sample_X, sample_y = rungs[0]
    print("sample rung:", sample_name)
    print(np.c_[sample_X, sample_y][:4])


def run_ladder(build_and_predict):
    rows = []
    for i, (name, X, y) in enumerate(clf_ladder(), start=1):
        acc = clf_accuracy(build_and_predict, X, y)
        rows.append((i, name, float(acc)))
    print("rung | accuracy | dataset")
    for i, name, acc in rows:
        print(f"D{i} | {acc:.3f} | {name}")
    return rows


def plot_summary(rows, model_factory, title):
    rungs = clf_ladder()
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    axes = axes.ravel()
    for ax, (name, X, y) in zip(axes[:5], rungs):
        _plot_regions(ax, model_factory(), X, y, name.split("(")[0])
    axes[5].plot([r[0] for r in rows], [r[2] for r in rows], marker="o")
    axes[5].set_ylim(0.0, 1.05)
    axes[5].set_xlabel("ladder rung")
    axes[5].set_ylabel("held-out accuracy")
    axes[5].set_title(title)
    axes[5].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def lesson_score(losses, cost, alternative):
    empirical = round(sum(losses) / len(losses), 3)
    score = round(empirical + cost, 3)
    gap = round(alternative - score, 3)
    return empirical, score, gap


def cost_sensitive_choice(raw_loss, cost, competitor):
    score = raw_loss + cost
    return score, score < competitor


## The concept, built once (D1)

The lesson formula is $$\tilde L=\sum_i(g_i f_i+\tfrac12 h_i f_i^2)+\Omega(f)$$. We first rebuild the tiny score: average the three lesson losses, add the cost, and assert the exact plan numbers.

In [ ]:

def xgboost_lightgbm_catboost_method():
    losses = np.array([0.18, 0.096, 0.42], dtype=float)
    empirical, score, gap = lesson_score(losses, 0.08, 0.356)
    assert empirical == 0.232
    assert score == 0.312
    assert gap == 0.044
    g = np.array([-0.35, 0.20, -0.10])
    h = np.array([0.24, 0.21, 0.25])
    f = np.array([0.50, -0.25, 0.10])
    omega = 0.03
    second_order = float(np.sum(g * f + 0.5 * h * f ** 2) + omega)
    return {"empirical": empirical, "score": score, "gap": gap, "second_order_surrogate": second_order}

result = xgboost_lightgbm_catboost_method()
print(result)


The printed dictionary contains the hand-checkable lesson score plus one method-specific quantity: a vote weight, an additive update, a second-order surrogate, a blend, a margin objective, or a kernel identity.

In [ ]:
checked = xgboost_lightgbm_catboost_method()
assert checked['score'] == 0.312
print('D1 arithmetic verified for 3.27')

## The dataset ladder

All classification notebooks use the shared `clf_ladder()` and `clf_accuracy()` helpers embedded above, so the notebook is self-contained in Colab.

In [ ]:
rungs = clf_ladder()
summarize_ladder(rungs)

## Run the same method across D1-D5

The metric is held-out accuracy. Macro-F1 would be a useful companion when class skew is severe, especially on D5.

In [ ]:


def build_and_predict(x_tr, y_tr, x_te):
    model = HistGradientBoostingClassifier(max_iter=90, learning_rate=0.08, l2_regularization=0.05, random_state=7)
    return _fit_predict(model, x_tr, y_tr, x_te)

rows = run_ladder(build_and_predict)
assert len(rows) == 5
assert all(0.0 <= acc <= 1.0 for _, _, acc in rows)


## Results visualization

The small multiples show the learned artifact on the first two standardized features for every rung; the summary panel tracks accuracy as the data become more realistic.

In [ ]:
plot_summary(rows, lambda: HistGradientBoostingClassifier(max_iter=90, learning_rate=0.08, l2_regularization=0.05, random_state=7), 'XGBoost / LightGBM / CatBoost accuracy')

## Pitfall on D5: optimizing the raw term and forgetting the cost

On D5, the unpenalized raw loss favors the larger boosted model. The lesson's regularization/cost term is the CPU-only stand-in for the production libraries' tree-complexity penalties. The wrong behavior ranks by raw validation loss; the fix ranks by the lesson score with cost and the validation-gap scale.

In [ ]:

X, y = clf_ladder()[-1][1:]
x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
scaler = StandardScaler()
x_tr = scaler.fit_transform(x_tr)
x_te = scaler.transform(x_te)
lean = HistGradientBoostingClassifier(max_iter=90, learning_rate=0.08, l2_regularization=0.05, random_state=7)
large = HistGradientBoostingClassifier(max_iter=90, learning_rate=0.08, l2_regularization=0.05, random_state=7)
if hasattr(large, "set_params"):
    params = large.get_params()
    if "n_estimators" in params:
        large.set_params(n_estimators=min(params["n_estimators"] * 3, 240))
    if "max_iter" in params and params["max_iter"] > 0:
        large.set_params(max_iter=min(params["max_iter"] * 3, 240))
    if "C" in params:
        large.set_params(C=25.0)
    if "gamma" in params:
        large.set_params(gamma=3.0)
lean.fit(x_tr, y_tr)
large.fit(x_tr, y_tr)
lean_loss = 1.0 - accuracy_score(y_te, lean.predict(x_te))
large_loss = 1.0 - accuracy_score(y_te, large.predict(x_te))
wrong_pick = "large" if large_loss <= lean_loss else "lean"
lean_score, lean_ok = cost_sensitive_choice(lean_loss, 0.08, large_loss + 0.08 + 0.044)
large_score = large_loss + 0.08 + 0.044
fixed_pick = "lean" if lean_score <= large_score else "large"
print("raw losses:", {"lean": round(lean_loss, 3), "large": round(large_loss, 3)})
print("wrong raw-loss pick:", wrong_pick)
print("cost-aware scores:", {"lean": round(lean_score, 3), "large": round(large_score, 3)})
print("fixed pick:", fixed_pick)
assert lean_score <= large_score or fixed_pick == "large"


## Evaluate it + Practice

- Compare held-out accuracy against a majority-class no-skill baseline.
- Sanity check that shuffling labels pushes accuracy toward chance.
- Ablate the key idea: fewer boosting rounds, no honest stacking, linear instead of kernel, or tiny/huge C should change the metric.
- Watch failure signals: unstable D5 score, perfect train accuracy with weak validation accuracy, or a cost-aware score that reverses the raw-loss winner.

Practice 1: change one hyperparameter and re-run the D1-D5 table.

Practice 2: add a majority-class baseline row for every rung.

Practice 3: repeat D5 with a different random split and compare the validation gap.